# Script 1 — Ingestão, Limpeza, Validação Temporal & EDA
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Pipeline em 6 etapas sequenciais:

| Etapa | Descrição |
|-------|-----------|
| 0 | Dependências, logging, configuração global |
| 1 | Inspeção de estrutura dos ZIPs (antes de qualquer transformação) |
| 2 | Ingestão — datetime nativo, timezone BR, sem `errors='coerce'` |
| 3 | Validação temporal — V1 datas futuras, V2 inconsistência ano, V3 duração período, V4 duplicatas, V5 intervalos |
| 4 | Pivot, D&A, KPIs financeiros, feature engineering temporal |
| 5 | Deduplicação final, EDA 9 blocos, persistência Parquet + auditoria |

> **Patches aplicados:** inspeção de estrutura · validação `DT_INI_EXERC` · fail-fast configurável  
> **Correções aplicadas:** CNPJ Vibra Energia · `sep=` explícito · pivot ordenado · merge `outer` com filtro · `VL_CONTA` sem `errors='coerce'`

## Etapa 0 — Dependências, logging e configuração global

In [5]:

import zipfile, logging, json, io
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo          # Python ≥ 3.9 (stdlib)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

# ── Timezone padrão Brasil ─────────────────────────────────────────────────────
# DECISÃO: todos os timestamps usam America/Sao_Paulo para garantir consistência
# em joins, serialização Parquet e downstream ML.
TZ_BRASIL   = ZoneInfo('America/Sao_Paulo')
AGORA_LOCAL = datetime.now(tz=TZ_BRASIL)

# ── Logging estruturado ────────────────────────────────────────────────────────
# Substitui print() por logging → rastreabilidade em pesquisa e produção.
# Handlers: terminal (StreamHandler) e arquivo (FileHandler) simultâneos.
logger = logging.getLogger('pipeline_cvm')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()  # evita handlers duplicados em re-execuções

_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)

# ── Parâmetros globais ─────────────────────────────────────────────────────────
PASTA_ZIPS_DFP  = Path('TCC_dados/DFP')
PASTA_ZIPS_ITR  = Path('TCC_dados/ITR')
PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

# CORREÇÃO B (documentada): sep e encoding são constantes separadas.
# A versão anterior tinha `sep=ENCODING_CVM and SEP_CVM` — expressão booleana
# que funcionava por acaso (string não-vazia é truthy). Agora explícito.
SEP_CVM      = ';'
ENCODING_CVM = 'latin1'

COLS_DATA_CVM    = ['DT_REFER', 'DT_INI_EXERC', 'DT_FIM_EXERC']
FORMATO_DATA_CVM = '%Y-%m-%d'   # ISO 8601 — formato oficial CVM

# ── PATCH 3 — Fail-fast configurável ──────────────────────────────────────────
# False (padrão TCC): pipeline continua mesmo com erros, apenas loga.
# True  (produção)  : lança ValueError se limiar de erros graves for atingido.
MODO_RIGOROSO           = False
LIMIAR_ERROS_DATA_PCT   = 0.05   # >5% datas inválidas → erro em modo rigoroso
LIMIAR_INCONSISTENCIAS  = 0.10   # >10% inconsistências ANO → erro em modo rigoroso

# ── Acumulador de auditoria ────────────────────────────────────────────────────
AUDITORIA: dict = {
    'zips_processados'         : [],
    'linhas_lidas_total'       : 0,
    'linhas_filtradas_anchor'  : 0,
    'erros_data'               : [],
    'datas_futuras'            : [],
    'inconsistencias_ano'      : [],
    'periodos_irregulares'     : [],   # PATCH 2 — duração de período anômala
    'duplicatas_removidas'     : 0,
    'registros_descartados'    : 0,
    'alertas_merge_outer'      : [],   # OBS D — linhas sem DRE+BPA após merge
}

logger.info("Pipeline iniciado | %s | Modo rigoroso: %s",
            AGORA_LOCAL.strftime('%Y-%m-%d %H:%M:%S %Z'), MODO_RIGOROSO)
logger.info("Pasta ZIPs DFP: %s", PASTA_ZIPS_DFP.resolve())
logger.info("Pasta ZIPs ITR: %s", PASTA_ZIPS_ITR.resolve())
logger.info("Pasta saída: %s", PASTA_SAIDA.resolve())


2026-03-29 18:46:19 | INFO     | Pipeline iniciado | 2026-03-29 18:46:19 -03 | Modo rigoroso: False
2026-03-29 18:46:19 | INFO     | Pasta ZIPs DFP: C:\Users\marce\OneDrive\Área de Trabalho\TCC\TCC\TCC_dados\DFP
2026-03-29 18:46:19 | INFO     | Pasta ZIPs ITR: C:\Users\marce\OneDrive\Área de Trabalho\TCC\TCC\TCC_dados\ITR
2026-03-29 18:46:19 | INFO     | Pasta saída: C:\Users\marce\OneDrive\Área de Trabalho\TCC\TCC\outputs


## Etapa 1A — Catálogo das 25 empresas âncora

In [6]:

# CORREÇÃO A: Vibra Energia estava com CNPJ idêntico ao da Raízen
# (33.453.598/0001-23), causando sobreposição de registros.
# CNPJ correto da Vibra: 04.628.902/0001-38.
EMPRESAS = {
    'Petróleo': {
        'Petrobras':     '33.000.167/0001-01',
        'Prio':          '10.629.105/0001-68',
        'Ultrapar':      '33.256.439/0001-39',
        'Raízen':        '33.453.598/0001-23',
        'Vibra Energia': '04.628.902/0001-38',  # ← CNPJ correto
    },
    'Energia': {
        'Engie Brasil':       '02.726.168/0001-97',
        'Equatorial Energia': '02.722.865/0001-82',
        'Taesa':              '07.859.971/0001-30',
        'CPFL Energia':       '02.429.144/0001-93',
        'ISA CTEEP':          '02.998.611/0001-04',
    },
    'Varejo': {
        'Lojas Renner':  '92.754.738/0001-62',
        'Magazine Luiza':'47.960.950/0001-21',
        'Alpargatas':    '61.079.117/0001-05',
        'Arezzo':        '16.590.234/0001-76',
        'Grupo Mateus':  '01.884.051/0001-92',
    },
    'Commodities': {
        'Vale':          '33.592.510/0001-54',
        'Suzano':        '16.404.287/0001-55',
        'Klabin':        '89.637.490/0001-45',
        'Gerdau':        '33.611.500/0001-19',
        'CSN Mineração': '33.042.730/0001-04',
    },
    'Tecnologia': {
        'WEG':       '84.429.695/0001-11',
        'Totvs':     '53.113.791/0001-22',
        'Positivo':  '81.243.735/0001-48',
        'Intelbras': '82.901.000/0001-27',
        'Brisanet':  '24.047.792/0001-60',
    },
}

# Garante unicidade de CNPJs no catálogo — detecta colisões antes de processar
def normalizar_cnpj(cnpj: str) -> str:
    """Remove pontuação: '33.000.167/0001-01' → '33000167000101'"""
    return ''.join(c for c in cnpj if c.isdigit())

cnpj_para_nome      : dict[str, str] = {}
cnpj_para_setor     : dict[str, str] = {}
nome_para_cnpj      : dict[str, str] = {}
cnpjnorm_para_nome  : dict[str, str] = {}
cnpjnorm_para_setor : dict[str, str] = {}

for setor, emps in EMPRESAS.items():
    for nome, cnpj in emps.items():
        cnorm = normalizar_cnpj(cnpj)
        # Detecta CNPJ duplicado no catálogo (ex: dois nomes para o mesmo CNPJ)
        if cnorm in cnpjnorm_para_nome:
            raise ValueError(
                f"CNPJ duplicado no catálogo: {cnpj} → '{nome}' e "
                f"'{cnpjnorm_para_nome[cnorm]}'. Corrija antes de prosseguir."
            )
        cnpj_para_nome[cnpj]       = nome
        cnpj_para_setor[cnpj]      = setor
        nome_para_cnpj[nome]       = cnpj
        cnpjnorm_para_nome[cnorm]  = nome
        cnpjnorm_para_setor[cnorm] = setor

TODOS_CNPJS_NORM = set(cnpjnorm_para_nome.keys())

logger.info("Catálogo: %d empresas únicas em %d setores",
            len(cnpjnorm_para_nome), len(EMPRESAS))
for setor, emps in EMPRESAS.items():
    logger.info("  %-15s %s", setor, list(emps.keys()))


2026-03-29 18:46:25 | INFO     | Catálogo: 25 empresas únicas em 5 setores
2026-03-29 18:46:25 | INFO     |   Petróleo        ['Petrobras', 'Prio', 'Ultrapar', 'Raízen', 'Vibra Energia']
2026-03-29 18:46:25 | INFO     |   Energia         ['Engie Brasil', 'Equatorial Energia', 'Taesa', 'CPFL Energia', 'ISA CTEEP']
2026-03-29 18:46:25 | INFO     |   Varejo          ['Lojas Renner', 'Magazine Luiza', 'Alpargatas', 'Arezzo', 'Grupo Mateus']
2026-03-29 18:46:25 | INFO     |   Commodities     ['Vale', 'Suzano', 'Klabin', 'Gerdau', 'CSN Mineração']
2026-03-29 18:46:25 | INFO     |   Tecnologia      ['WEG', 'Totvs', 'Positivo', 'Intelbras', 'Brisanet']


## Etapa 1B — Inspeção de estrutura dos ZIPs (PATCH 1)

Executar **antes** de qualquer transformação para confirmar que os dados brutos estão no formato esperado.

In [8]:

# PATCH 1 — Inspeção explícita da estrutura dos arquivos CVM
# Exibe: lista de arquivos no ZIP, colunas + dtypes de cada CSV,
# amostra de 3 linhas e cardinalidade das colunas-chave.
# Isso documenta o formato bruto dos dados no TCC e detecta mudanças
# de estrutura entre anos (a CVM já alterou colunas em versões passadas).

def inspecionar_zip(caminho_zip: Path, max_arquivos: int = 5) -> None:
    """
    Inspeciona a estrutura de um ZIP CVM sem carregar os dados completos.

    Parâmetros
    ----------
    caminho_zip  : caminho para o arquivo .zip
    max_arquivos : quantos CSVs inspecionar (evita output excessivo em ZIPs grandes)
    """
    print(f"\n{'='*70}")
    print(f"📦 ZIP: {caminho_zip.name}")
    print(f"{'='*70}")

    with zipfile.ZipFile(caminho_zip) as z:
        csvs = [n for n in z.namelist() if n.endswith('.csv')]
        print(f"  Arquivos CSV encontrados: {len(csvs)}")
        for nome in csvs[:max_arquivos]:
            print(f"\n  📄 {nome}")
            with z.open(nome) as f:
                # Lê apenas as primeiras 200 linhas para inspeção rápida
                amostra = pd.read_csv(
                    f, sep=SEP_CVM, encoding=ENCODING_CVM,
                    nrows=200, low_memory=False,
                    dtype=str,   # tudo como string — inspeção apenas
                )
            print(f"     Colunas ({len(amostra.columns)}):")
            for col in amostra.columns:
                n_nulos = amostra[col].isna().sum()
                n_uniq  = amostra[col].nunique()
                print(f"       - {col:<30} | nulos: {n_nulos:>3} | únicos: {n_uniq:>4}")
            print(f"\n     Amostra (3 linhas):")
            print(amostra.head(3).to_string(index=False))
        if len(csvs) > max_arquivos:
            print(f"\n  ... e mais {len(csvs) - max_arquivos} arquivos não exibidos.")

# Executa inspeção nos ZIPs disponíveis (limite de 1 ZIP para não poluir output)
zips_disponiveis_dfp = sorted(PASTA_ZIPS_DFP.glob('dfp_cia_aberta_*.zip'))

if zips_disponiveis_dfp:
    # Inspeciona o ZIP mais recente como representativo
    inspecionar_zip(zips_disponiveis_dfp[-1], max_arquivos=3)
    logger.info("Inspeção concluída — %d ZIPs disponíveis: %s",
                len(zips_disponiveis_dfp), [z.name for z in zips_disponiveis_dfp])
else:
    logger.warning("Nenhum ZIP encontrado em '%s'. Configure PASTA_ZIPS_DFP.", PASTA_ZIPS_DFP.resolve())
    
# Executa inspeção nos ZIPs disponíveis (limite de 1 ZIP para não poluir output)
zips_disponiveis_itr = sorted(PASTA_ZIPS_ITR.glob('itr_cia_aberta_*.zip'))

if zips_disponiveis_itr:
    # Inspeciona o ZIP mais recente como representativo
    inspecionar_zip(zips_disponiveis_itr[-1], max_arquivos=3)
    logger.info("Inspeção concluída — %d ZIPs disponíveis: %s",
                len(zips_disponiveis_itr), [z.name for z in zips_disponiveis_itr])
else:
    logger.warning("Nenhum ZIP encontrado em '%s'. Configure PASTA_ZIPS_ITR.", PASTA_ZIPS_ITR.resolve())


2026-03-29 18:48:16 | INFO     | Inspeção concluída — 5 ZIPs disponíveis: ['dfp_cia_aberta_2021.zip', 'dfp_cia_aberta_2022.zip', 'dfp_cia_aberta_2023.zip', 'dfp_cia_aberta_2024.zip', 'dfp_cia_aberta_2025.zip']
2026-03-29 18:48:16 | INFO     | Inspeção concluída — 5 ZIPs disponíveis: ['itr_cia_aberta_2021.zip', 'itr_cia_aberta_2022.zip', 'itr_cia_aberta_2023.zip', 'itr_cia_aberta_2024.zip', 'itr_cia_aberta_2025.zip']



📦 ZIP: dfp_cia_aberta_2025.zip
  Arquivos CSV encontrados: 19

  📄 dfp_cia_aberta_2025.csv
     Colunas (9):
       - CNPJ_CIA                       | nulos:   0 | únicos:   33
       - DT_REFER                       | nulos:   0 | únicos:    4
       - VERSAO                         | nulos:   0 | únicos:    2
       - DENOM_CIA                      | nulos:   0 | únicos:   33
       - CD_CVM                         | nulos:   0 | únicos:   33
       - CATEG_DOC                      | nulos:   0 | únicos:    1
       - ID_DOC                         | nulos:   0 | únicos:   39
       - DT_RECEB                       | nulos:   0 | únicos:   16
       - LINK_DOC                       | nulos:   0 | únicos:   39

     Amostra (3 linhas):
          CNPJ_CIA   DT_REFER VERSAO                              DENOM_CIA CD_CVM CATEG_DOC ID_DOC   DT_RECEB                                                                                                              LINK_DOC
01.027.058/0001-91 2025

## Etapa 2 — Ingestão com datetime nativo e timezone BR

In [9]:

# ── Função de parse de datas — SEM errors='coerce' silencioso ─────────────────
def _parse_coluna_data(serie: pd.Series, col: str, arquivo: str) -> pd.Series:
    """
    Converte coluna de texto para datetime com timezone Brasil.

    Estratégia explícita (sem errors='coerce'):
      1. Parse vetorial com formato oficial CVM ('%Y-%m-%d').
      2. Valores que falham são detectados por máscara e logados individualmente.
      3. Aplica tz_localize('America/Sao_Paulo') → timestamps timezone-aware.
         Isso evita problemas em joins, serialização Parquet e comparações.
    """
    # Parse vetorial — errors='coerce' aqui é intencional e rastreado logo abaixo
    resultado = pd.to_datetime(serie, format=FORMATO_DATA_CVM, errors='coerce')

    # Detecta e registra explicitamente cada valor inválido
    mask_inv = resultado.isna() & serie.notna() & (serie.str.strip() != '')
    if mask_inv.any():
        for idx, val in serie[mask_inv].items():
            reg = {'arquivo': arquivo, 'coluna': col, 'valor': val, 'indice': int(idx)}
            AUDITORIA['erros_data'].append(reg)
            logger.warning("Data inválida | arq=%-30s col=%-15s val=%r", arquivo, col, val)

        # PATCH 3 — Fail-fast: verifica limiar em modo rigoroso
        if MODO_RIGOROSO:
            taxa = mask_inv.sum() / max(len(serie), 1)
            if taxa > LIMIAR_ERROS_DATA_PCT:
                raise ValueError(
                    f"[MODO_RIGOROSO] {taxa:.1%} de datas inválidas em {arquivo}/{col} "
                    f"(limiar: {LIMIAR_ERROS_DATA_PCT:.0%})"
                )

    # Aplica timezone Brasil — todos os timestamps se tornam timezone-aware
    # nonexistent='shift_forward': trata hora inexistente no horário de verão
    # ambiguous='infer': trata hora ambígua na mudança de horário
    resultado = resultado.dt.tz_localize(
        TZ_BRASIL, ambiguous='infer', nonexistent='shift_forward'
    )
    return resultado


def ler_csv_cvm(zip_path: Path, nome_arquivo: str) -> pd.DataFrame:
    """
    Lê um CSV dentro de um ZIP CVM com todas as correções aplicadas:
    - sep=';', encoding='latin1'  (CORREÇÃO B — sep explícito, não booleano)
    - Datetime convertido imediatamente após leitura (não após merge)
    - VL_CONTA lido como str e convertido com validação explícita (CORREÇÃO E)
    """
    with zipfile.ZipFile(zip_path) as z:
        matches = [n for n in z.namelist() if nome_arquivo in n]
        if not matches:
            return pd.DataFrame()
        nome_interno = matches[0]
        with z.open(nome_interno) as f:
            df = pd.read_csv(
                f,
                sep=SEP_CVM,           # CORREÇÃO B: explícito, não booleano
                encoding=ENCODING_CVM,
                dtype={
                    'CNPJ_CIA'  : str,
                    'CD_CVM'    : str,
                    'CD_CONTA'  : str,
                    'VERSAO'    : 'Int64',
                    'VL_CONTA'  : str,   # CORREÇÃO E: lido como str (evita notação científica)
                    'ORDEM_EXERC': str,
                },
                low_memory=False,
            )

    # ── Conversão de datas IMEDIATAMENTE após leitura ──────────────────────────
    # DECISÃO: não acumular DataFrames com datas como string. Merges posteriores
    # usam datetime nativo → performance e consistência garantidas.
    for col in COLS_DATA_CVM:
        if col in df.columns:
            df[col] = _parse_coluna_data(df[col], col, nome_interno)

    # ── CORREÇÃO E: VL_CONTA — conversão explícita sem errors='coerce' ─────────
    if 'VL_CONTA' in df.columns:
        vl_num = pd.to_numeric(df['VL_CONTA'], errors='coerce')
        # Detecta valores que não converteram (exceto strings vazias/nulas)
        mask_vl_inv = vl_num.isna() & df['VL_CONTA'].notna() & (df['VL_CONTA'].str.strip() != '')
        if mask_vl_inv.any():
            n_inv = mask_vl_inv.sum()
            logger.warning("VL_CONTA inválido | %s | %d valores não numéricos", nome_interno, n_inv)
        df['VL_CONTA'] = vl_num

    return df


def carregar_demonstrativo(tipo: str, modalidade: str = 'con') -> pd.DataFrame:
    """
    Carrega e concatena todos os ZIPs para um tipo de demonstrativo.
    Filtra para as 25 empresas âncora e enriquece com metadados.
    """
    nome_arquivo = f'dfp_cia_aberta_{tipo}_{modalidade}_'
    zips = sorted(PASTA_ZIPS.glob('dfp_cia_aberta_*.zip'))
    if not zips:
        logger.error("Nenhum ZIP em '%s'", PASTA_ZIPS)
        return pd.DataFrame()

    partes = []
    for zp in zips:
        df = ler_csv_cvm(zp, nome_arquivo)
        if df.empty:
            continue
        n_antes = len(df)
        AUDITORIA['linhas_lidas_total'] += n_antes

        df = df.copy()
        df['CNPJ_NORM'] = df['CNPJ_CIA'].apply(normalizar_cnpj)
        df = df[df['CNPJ_NORM'].isin(TODOS_CNPJS_NORM)]
        n_depois = len(df)

        AUDITORIA['linhas_filtradas_anchor'] += n_depois
        AUDITORIA['registros_descartados']   += (n_antes - n_depois)
        logger.info("  %s | %s: %d → %d linhas", zp.name, tipo, n_antes, n_depois)

        if not df.empty:
            partes.append(df)
            if zp.name not in AUDITORIA['zips_processados']:
                AUDITORIA['zips_processados'].append(zp.name)

    if not partes:
        logger.warning("Sem dados para %s_%s", tipo, modalidade)
        return pd.DataFrame()

    dfinal = pd.concat(partes, ignore_index=True)
    dfinal['NOME_CIA'] = dfinal['CNPJ_NORM'].map(cnpjnorm_para_nome)
    dfinal['SETOR']    = dfinal['CNPJ_NORM'].map(cnpjnorm_para_setor)
    dfinal['TIPO_DOC'] = tipo
    if 'DT_REFER' in dfinal.columns:
        dfinal['ANO_REF'] = dfinal['DT_REFER'].dt.year

    # Deduplicação de versão: mantém VERSAO mais alta (retificação mais recente)
    chave_v = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA']
    if 'ORDEM_EXERC' in dfinal.columns:
        chave_v.append('ORDEM_EXERC')
    if 'VERSAO' in dfinal.columns:
        n_antes = len(dfinal)
        dfinal = (dfinal
                  .sort_values(['CNPJ_CIA','DT_REFER','VERSAO'],
                               ascending=[True,True,False])
                  .drop_duplicates(subset=chave_v, keep='first'))
        rem = n_antes - len(dfinal)
        AUDITORIA['duplicatas_removidas'] += rem
        if rem:
            logger.info("  %s: %d duplicatas de versão removidas", tipo, rem)

    anos = sorted(dfinal['ANO_REF'].dropna().astype(int).unique()) if 'ANO_REF' in dfinal.columns else []
    logger.info("%s_%s: %d linhas | %d empresas | anos %s",
                tipo, modalidade, len(dfinal), dfinal['NOME_CIA'].nunique(), anos)
    return dfinal


## Etapa 2B — Execução do carregamento

In [ ]:

logger.info("=== Carregamento dos demonstrativos ===")
dre = carregar_demonstrativo('DRE',    'con')
bpa = carregar_demonstrativo('BPA',    'con')
bpp = carregar_demonstrativo('BPP',    'con')
dfc = carregar_demonstrativo('DFC_MI', 'con')
dva = carregar_demonstrativo('DVA',    'con')
logger.info("=== Carregamento concluído | erros de data: %d ===",
            len(AUDITORIA['erros_data']))


## Etapa 3 — Validação temporal (5 verificações)

In [ ]:

# ── Limites para DFP anual ─────────────────────────────────────────────────────
# PATCH 2: usamos DT_INI_EXERC e DT_FIM_EXERC para calcular duração real do período.
# DFP anual normal: 330–400 dias (tolerância para exercícios fiscais atípicos).
# Empresas com fiscal year fora do calendário civil (ex: Raízen: abr→mar = 365 dias)
# são DETECTADAS mas não descartadas — apenas registradas em AUDITORIA.
DURACAO_DFP_MIN_DIAS = 300
DURACAO_DFP_MAX_DIAS = 400

# Limite para intervalos entre DFPs consecutivos da mesma empresa
INTERVALO_DFP_MIN = 300   # menos de 300 dias entre DFPs → possível ITR misturado
INTERVALO_DFP_MAX = 400   # mais de 400 dias → gap temporal suspeito

def validar_qualidade_temporal(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """
    Executa 5 validações temporais e retorna o DataFrame limpo.

    V1 — Datas futuras (DT_REFER > agora) → REMOVE
    V2 — Inconsistência ANO (DT_FIM_EXERC.year ≠ ANO_REF) → LOGA, mantém
    V3 — Duração do período (DT_FIM_EXERC - DT_INI_EXERC) fora de 300–400 dias → LOGA  [PATCH 2]
    V4 — Duplicatas por (CNPJ_CIA, DT_FIM_EXERC, CD_CONTA) → REMOVE
    V5 — Intervalos irregulares entre DFPs consecutivos → LOGA
    """
    if df.empty:
        return df

    df = df.copy()
    n_original = len(df)
    agora_tz   = pd.Timestamp(AGORA_LOCAL)

    # ── V1: Datas futuras ──────────────────────────────────────────────────────
    col_ref = 'DT_REFER' if 'DT_REFER' in df.columns else None
    if col_ref:
        mask_fut = df[col_ref].notna() & (df[col_ref] > agora_tz)
        n_fut = mask_fut.sum()
        if n_fut:
            exemplos = df.loc[mask_fut, ['CNPJ_CIA', col_ref]].head(5).to_dict('records')
            logger.warning("V1 | %s | %d datas futuras removidas | ex: %s",
                           nome, n_fut, exemplos)
            AUDITORIA['datas_futuras'].extend(exemplos)
            df = df[~mask_fut].copy()
            AUDITORIA['registros_descartados'] += n_fut

            # PATCH 3 — Fail-fast
            if MODO_RIGOROSO:
                taxa = n_fut / max(n_original, 1)
                if taxa > LIMIAR_INCONSISTENCIAS:
                    raise ValueError(
                        f"[MODO_RIGOROSO] {taxa:.1%} de datas futuras em {nome}"
                    )

    # ── V2: DT_FIM_EXERC.year ≠ ANO_REF ──────────────────────────────────────
    if 'DT_FIM_EXERC' in df.columns and 'ANO_REF' in df.columns:
        mask_v2 = (
            df['DT_FIM_EXERC'].notna() & df['ANO_REF'].notna() &
            (df['DT_FIM_EXERC'].dt.year != df['ANO_REF'])
        )
        n_v2 = mask_v2.sum()
        if n_v2:
            exemplos = df.loc[mask_v2, ['CNPJ_CIA','DT_FIM_EXERC','ANO_REF']].head(5).to_dict('records')
            # DECISÃO: NÃO remove — pode ser exercício fiscal irregular (ex: Raízen)
            logger.warning("V2 | %s | %d inconsistências DT_FIM_EXERC vs ANO_REF (mantidos) | ex: %s",
                           nome, n_v2, exemplos)
            AUDITORIA['inconsistencias_ano'].extend(exemplos)
            if MODO_RIGOROSO:
                taxa = n_v2 / max(len(df), 1)
                if taxa > LIMIAR_INCONSISTENCIAS:
                    raise ValueError(
                        f"[MODO_RIGOROSO] {taxa:.1%} de inconsistências ANO em {nome}"
                    )

    # ── V3: Duração do período (PATCH 2 — usa DT_INI_EXERC) ──────────────────
    if 'DT_INI_EXERC' in df.columns and 'DT_FIM_EXERC' in df.columns:
        mask_datas = df['DT_INI_EXERC'].notna() & df['DT_FIM_EXERC'].notna()
        if mask_datas.any():
            duracao = (df.loc[mask_datas, 'DT_FIM_EXERC'] -
                       df.loc[mask_datas, 'DT_INI_EXERC']).dt.days
            mask_curta  = duracao < DURACAO_DFP_MIN_DIAS
            mask_longa  = duracao > DURACAO_DFP_MAX_DIAS
            mask_anomala = mask_curta | mask_longa

            if mask_anomala.any():
                idx_anomalos = duracao[mask_anomala].index
                sample = df.loc[idx_anomalos[:5],
                                ['CNPJ_CIA','NOME_CIA','DT_INI_EXERC',
                                 'DT_FIM_EXERC']].copy()
                sample['duracao_dias'] = duracao[mask_anomala[:5]].values
                registros = sample.to_dict('records')
                logger.warning(
                    "V3 | %s | %d períodos com duração anômala "
                    "(<300d ou >400d) — fiscal year atípico? | ex: %s",
                    nome, mask_anomala.sum(), registros
                )
                AUDITORIA['periodos_irregulares'].extend(registros)
                # DECISÃO: NÃO remove — fiscal year irregular é válido (Raízen, etc.)

    # ── V4: Duplicatas por (CNPJ_CIA, DT_FIM_EXERC, CD_CONTA) ────────────────
    if 'DT_FIM_EXERC' in df.columns:
        chave_dup = ['CNPJ_CIA', 'DT_FIM_EXERC', 'CD_CONTA']
        if 'ORDEM_EXERC' in df.columns:
            chave_dup.append('ORDEM_EXERC')
        n_antes_v4 = len(df)
        # DECISÃO: keep='last' — em caso de duplicata, mantém o registro mais recente
        df = df.drop_duplicates(subset=chave_dup, keep='last')
        rem_v4 = n_antes_v4 - len(df)
        if rem_v4:
            AUDITORIA['duplicatas_removidas'] += rem_v4
            logger.info("V4 | %s | %d duplicatas (CNPJ+DT_FIM+CD_CONTA) removidas", nome, rem_v4)

    # ── V5: Intervalos irregulares entre DFPs consecutivos ────────────────────
    if 'DT_REFER' in df.columns and 'CNPJ_CIA' in df.columns:
        df_ord = (df[['CNPJ_CIA','DT_REFER']].drop_duplicates()
                  .sort_values(['CNPJ_CIA','DT_REFER']))
        df_ord['_delta'] = df_ord.groupby('CNPJ_CIA')['DT_REFER'].diff().dt.days
        irr = df_ord[df_ord['_delta'].notna() &
                     ((df_ord['_delta'] < INTERVALO_DFP_MIN) |
                      (df_ord['_delta'] > INTERVALO_DFP_MAX))]
        if not irr.empty:
            logger.warning(
                    "V5 | %s | %d intervalos irregulares entre DFPs:\n%s",
                    nome, len(irr),
                    irr[['CNPJ_CIA','DT_REFER','_delta']].to_string()
                )

    logger.info("Validação | %s | %d → %d linhas (-%d)",
                nome, n_original, len(df), n_original - len(df))
    return df

logger.info("=== Validação temporal ===")
dre = validar_qualidade_temporal(dre, 'DRE')
bpa = validar_qualidade_temporal(bpa, 'BPA')
bpp = validar_qualidade_temporal(bpp, 'BPP')
dfc = validar_qualidade_temporal(dfc, 'DFC_MI')


## Etapa 4A — Pivotagem (formato longo → largo)

In [ ]:

# CORREÇÃO C: pivot_table com aggfunc='last' dependia da ordem das linhas.
# Solução: ordenação explícita por VERSAO ANTES do pivot → determinístico.

def pivotar(df: pd.DataFrame, sufixo: str) -> pd.DataFrame:
    """
    Transforma formato longo CVM em formato largo.
    Filtra ORDEM_EXERC = 'ÚLTIMO' (valor corrente, não comparativo).
    Ordena por VERSAO antes do pivot para garantir resultado determinístico. (CORREÇÃO C)
    """
    if df.empty:
        return pd.DataFrame()

    if 'ORDEM_EXERC' in df.columns:
        df = df[df['ORDEM_EXERC'] == 'ÚLTIMO'].copy()

    # CORREÇÃO C — ordenação explícita antes do pivot
    sort_cols = ['CNPJ_CIA', 'DT_REFER']
    if 'VERSAO' in df.columns:
        sort_cols.append('VERSAO')
    df = df.sort_values(sort_cols, ascending=True)

    df['CD_CONTA'] = df['CD_CONTA'].astype(str).str.strip()
    df['VL_CONTA'] = pd.to_numeric(df['VL_CONTA'], errors='coerce')

    chave_idx = [c for c in ['CNPJ_CIA','NOME_CIA','SETOR','ANO_REF','DT_REFER']
                 if c in df.columns]

    pivot = df.pivot_table(
        index=chave_idx,
        columns='CD_CONTA',
        values='VL_CONTA',
        aggfunc='last',   # determinístico após sort_values acima
    )
    pivot.columns = [f'{sufixo}_{c}' for c in pivot.columns]
    pivot.columns.name = None
    result = pivot.reset_index()
    logger.info("Pivot %-5s: %d linhas × %d colunas", sufixo, *result.shape)
    return result

logger.info("=== Pivotagem ===")
p_dre = pivotar(dre, 'DRE')
p_bpa = pivotar(bpa, 'BPA')
p_bpp = pivotar(bpp, 'BPP')
p_dfc = pivotar(dfc, 'DFC')

# ── Merge progressivo com chave explícita ─────────────────────────────────────
# CORREÇÃO D — parte 1: usa how='outer' mas rastreia linhas sem DRE+BPA.
# O outer é necessário para capturar empresas com cobertura parcial de demonstrativos.
# O filtro pós-merge (aplicado adiante) garante qualidade mínima para ML.
CHAVE_MERGE = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO_REF', 'DT_REFER']

dataset = p_dre.copy() if not p_dre.empty else pd.DataFrame()
for p_df, nome_p in [(p_bpa,'BPA'), (p_bpp,'BPP'), (p_dfc,'DFC')]:
    if p_df.empty:
        logger.warning("Pivot %s vazio — merge ignorado", nome_p)
        continue
    chave = [c for c in CHAVE_MERGE if c in dataset.columns and c in p_df.columns]
    if dataset.empty:
        dataset = p_df.copy()
    else:
        n_antes = len(dataset)
        dataset = dataset.merge(p_df, on=chave, how='outer',
                                suffixes=('', f'__{nome_p}'))
        # Remove colunas duplicadas acidentais geradas pelo merge
        cols_dup = [c for c in dataset.columns if f'__{nome_p}' in c]
        if cols_dup:
            dataset.drop(columns=cols_dup, inplace=True)
        logger.info("Merge %-5s: %d → %d linhas", nome_p, n_antes, len(dataset))

logger.info("Dataset pós-pivot: %d × %d", *dataset.shape)


## Etapa 4B — Extração de D&A via DFC Método Indireto

In [ ]:

# DECISÃO METODOLÓGICA: D&A não é linha autônoma no DRE padrão CVM.
# Está embutida em CPV (3.02) e Despesas Operacionais (3.04).
# Extração via subcontas 6.01.01.xx do DFC_MI (busca por palavra-chave em DS_CONTA).
# Limitação documentada: empresas com apenas DFC_MD → DNA = NaN → EBITDA = NaN.

def extrair_dna(dfc_df: pd.DataFrame) -> pd.DataFrame:
    if dfc_df.empty:
        return pd.DataFrame()

    PALAVRAS_DNA = ['deprecia', 'amortiza', 'exaust', 'depletion']
    mask = (
        dfc_df['DS_CONTA'].str.lower().str.contains('|'.join(PALAVRAS_DNA), na=False) &
        (dfc_df['ORDEM_EXERC'] == 'ÚLTIMO' if 'ORDEM_EXERC' in dfc_df.columns else True)
    )
    dfc_dna = dfc_df[mask].copy()
    dfc_dna['VL_CONTA'] = pd.to_numeric(dfc_dna['VL_CONTA'], errors='coerce').abs()
    dfc_dna['DNA'] = dfc_dna['VL_CONTA'].replace(0, np.nan)  # 0 espúrio → NaN

    chave = [c for c in ['CNPJ_CIA','NOME_CIA','SETOR','ANO_REF','DT_REFER']
             if c in dfc_dna.columns]
    dna = dfc_dna.groupby(chave)['DNA'].sum().reset_index()
    dna['DNA'] = dna['DNA'].replace(0, np.nan)

    cobertura = dna['DNA'].notna().mean()
    logger.info("D&A: %d registros | %d empresas | cobertura %.0f%%",
                len(dna), dna['NOME_CIA'].nunique() if 'NOME_CIA' in dna.columns else 0,
                cobertura * 100)
    return dna

dna = extrair_dna(dfc)
if not dna.empty and not dataset.empty:
    chave_dna = [c for c in ['CNPJ_CIA','ANO_REF','DT_REFER']
                 if c in dataset.columns and c in dna.columns]
    dataset = dataset.merge(dna[chave_dna + ['DNA']], on=chave_dna, how='left')
    logger.info("Dataset com D&A: %d×%d | cobertura DNA: %.0f%%",
                *dataset.shape, dataset['DNA'].notna().mean() * 100)


## Etapa 4C — Cálculo dos 15 KPIs Financeiros

In [ ]:

LISTA_KPIS = [
    'margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
    'roe','roa','liquidez_corrente','liquidez_imediata',
    'endividamento','alavancagem_de','div_liquida','cobertura_juros',
    'giro_ativo','fco_receita','fco_lucro','EBITDA',
]

def calcular_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula 15 KPIs. Divisões por zero → NaN (sem Inf, sem warnings)."""
    d = df.copy()

    def get(p, c):
        col = f'{p}_{c}'
        return d[col].copy() if col in d.columns else pd.Series(np.nan, index=d.index)

    def _div(num, den):
        """Divisão segura: substitui zero no denominador por NaN antes de dividir."""
        return num / den.replace(0, np.nan)

    receita     = get('DRE','3.01');  lucro_bruto = get('DRE','3.03')
    ebit        = get('DRE','3.05');  lucro_liq   = get('DRE','3.11')
    desp_fin    = get('DRE','3.06');  ativo_tot   = get('BPA','1')
    ativo_circ  = get('BPA','1.01'); caixa       = get('BPA','1.01.01')
    pass_circ   = get('BPP','2.01'); div_cp      = get('BPP','2.01.04')
    div_lp      = get('BPP','2.02.01'); pat_liq   = get('BPP','2.03')
    fco         = get('DFC','6.01')
    dna         = d['DNA'].copy() if 'DNA' in d.columns else pd.Series(0.0, index=d.index)

    div_bruta = div_cp.fillna(0) + div_lp.fillna(0)
    ebitda    = ebit + dna.fillna(0)

    d['margem_bruta']      = _div(lucro_bruto, receita)
    d['margem_ebit']       = _div(ebit, receita)
    d['margem_liquida']    = _div(lucro_liq, receita)
    d['margem_ebitda']     = _div(ebitda, receita)
    d['roe']               = _div(lucro_liq, pat_liq)
    d['roa']               = _div(lucro_liq, ativo_tot)
    d['liquidez_corrente'] = _div(ativo_circ, pass_circ)
    d['liquidez_imediata'] = _div(caixa, pass_circ)
    d['endividamento']     = _div(div_bruta, ativo_tot)
    d['alavancagem_de']    = _div(div_bruta, pat_liq)
    d['div_liquida']       = div_bruta - caixa.fillna(0)
    d['cobertura_juros']   = _div(ebit, desp_fin.abs())
    d['giro_ativo']        = _div(receita, ativo_tot)
    d['fco_receita']       = _div(fco, receita)
    d['fco_lucro']         = _div(fco, lucro_liq)
    d['EBITDA']            = ebitda

    logger.info("KPIs calculados — cobertura:")
    for kpi in LISTA_KPIS:
        if kpi in d.columns:
            cob = d[kpi].notna().mean()
            nivel = "✅" if cob >= 0.5 else "⚠️"
            logger.info("  %s %-25s %.0f%%", nivel, kpi, cob * 100)
    return d

if not dataset.empty:
    dataset = calcular_kpis(dataset)
    logger.info("Dataset com KPIs: %d×%d", *dataset.shape)


## Etapa 4D — Feature Engineering Temporal (8 features)

In [ ]:

def engenharia_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria 8 features temporais derivadas de DT_REFER (já timezone-aware).
    Deve ser executada ANTES do Script 2 pois algumas features
    (DELTA_DIAS, PERIODO_ORDINAL) são usadas como features de ML.

    Features:
      ANO, TRIMESTRE, MES        — calendário básico
      ANO_TRIMESTRE              — 'YYYYQn', ex: '2024Q4'
      FLAG_FIM_ANO               — 1 se mês ∈ {10,11,12}
      FLAG_TRIMESTRE             — 1 se mês ∈ {3,6,9,12}
      DELTA_DIAS                 — dias entre períodos consecutivos da empresa
      PERIODO_ORDINAL            — rank temporal por empresa (1 = mais antigo)
    """
    if df.empty or 'DT_REFER' not in df.columns:
        logger.warning("Feature engineering: DT_REFER ausente — ignorado")
        return df

    d = df.sort_values(['CNPJ_CIA','DT_REFER']).copy()
    dt = d['DT_REFER'].dt

    d['ANO']            = dt.year.astype('Int64')
    d['TRIMESTRE']      = dt.quarter.astype('Int64')
    d['MES']            = dt.month.astype('Int64')
    d['ANO_TRIMESTRE']  = dt.year.astype(str) + 'Q' + dt.quarter.astype(str)
    d['FLAG_FIM_ANO']   = dt.month.isin([10,11,12]).astype('Int8')
    d['FLAG_TRIMESTRE'] = dt.month.isin([3,6,9,12]).astype('Int8')

    d['DELTA_DIAS'] = (
        d.groupby('CNPJ_CIA')['DT_REFER']
        .diff().dt.days.astype('Int64')
    )
    d['PERIODO_ORDINAL'] = (
        d.groupby('CNPJ_CIA').cumcount() + 1
    ).astype('Int64')

    logger.info("Features temporais: ANO, TRIMESTRE, MES, ANO_TRIMESTRE, "
                "FLAG_FIM_ANO, FLAG_TRIMESTRE, DELTA_DIAS, PERIODO_ORDINAL")
    return d

if not dataset.empty:
    dataset = engenharia_temporal(dataset)
    logger.info("Dataset após feature engineering: %d×%d", *dataset.shape)


## Etapa 5A — Deduplicação final e filtro de qualidade pós-merge

In [ ]:

if not dataset.empty:
    # ── Deduplicação final ─────────────────────────────────────────────────────
    # Chave de unicidade do dataset consolidado: (CNPJ_CIA, DT_REFER).
    # Critério de desempate: mantém o registro com mais KPIs válidos.
    kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
    n_antes_dedup  = len(dataset)

    dataset['_n_kpis'] = dataset[kpis_presentes].notna().sum(axis=1)
    dataset = (dataset
               .sort_values(['CNPJ_CIA','DT_REFER','_n_kpis'],
                            ascending=[True,True,False])
               .drop_duplicates(subset=['CNPJ_CIA','DT_REFER'], keep='first')
               .drop(columns=['_n_kpis'])
               .reset_index(drop=True))
    rem_dedup = n_antes_dedup - len(dataset)
    AUDITORIA['duplicatas_removidas'] += rem_dedup
    logger.info("Deduplicação final: %d → %d (-%d por CNPJ+DT_REFER)",
                n_antes_dedup, len(dataset), rem_dedup)

    # ── CORREÇÃO D — filtro pós-merge outer ───────────────────────────────────
    # O merge outer pode criar linhas onde uma empresa tem BPA mas não DRE
    # (dados parcialmente disponíveis). Essas linhas têm vetores de KPIs quase
    # inteiramente NaN e contaminam o treinamento ML.
    # DECISÃO: exige ao menos DRE_3.01 (Receita) preenchido como indicador
    # de que o registro tem informação financeira útil.
    # Registros filtrados são contabilizados em AUDITORIA.
    if 'DRE_3.01' in dataset.columns:
        n_antes_filtro = len(dataset)
        mask_sem_dre = dataset['DRE_3.01'].isna()
        n_sem_dre = mask_sem_dre.sum()
        if n_sem_dre > 0:
            registros_filtrados = dataset[mask_sem_dre][['CNPJ_CIA','NOME_CIA','DT_REFER']].to_dict('records')
            AUDITORIA['alertas_merge_outer'].extend(registros_filtrados[:20])
            logger.warning("CORREÇÃO D | %d linhas sem DRE_3.01 pós-merge outer — removidas",
                           n_sem_dre)
            dataset = dataset[~mask_sem_dre].reset_index(drop=True)
            AUDITORIA['registros_descartados'] += n_sem_dre

    logger.info("Dataset final consolidado: %d × %d", *dataset.shape)


## Etapa 5B — Análise Exploratória de Dados (9 Blocos)

In [ ]:

if dataset.empty:
    logger.warning("Dataset vazio — EDA ignorada.")
else:
    kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]

    print("=" * 70)
    print("BLOCO 1 — Visão Geral do Dataset")
    print("=" * 70)
    print(f"  Linhas           : {len(dataset):,}")
    print(f"  Colunas          : {dataset.shape[1]}")
    print(f"  Empresas         : {dataset['NOME_CIA'].nunique()}")
    print(f"  Setores          : {dataset['SETOR'].nunique()}")
    if 'ANO' in dataset.columns:
        anos = sorted(dataset['ANO'].dropna().astype(int).unique())
        print(f"  Anos             : {anos}")
    print(f"  Nulos global     : {dataset.isnull().mean().mean():.1%}")
    print(f"  Erros de data    : {len(AUDITORIA['erros_data'])}")
    print(f"  Datas futuras    : {len(AUDITORIA['datas_futuras'])}")
    print(f"  Incons. ano      : {len(AUDITORIA['inconsistencias_ano'])}")
    print(f"  Períodos irregul.: {len(AUDITORIA['periodos_irregulares'])}")
    print(f"  Duplicatas rem.  : {AUDITORIA['duplicatas_removidas']}")


In [ ]:

if not dataset.empty:
    print("BLOCO 2 — Cobertura Temporal por Empresa")
    cob = (dataset.groupby('NOME_CIA')
           .agg(Primeiro=('ANO','min'), Último=('ANO','max'),
                Períodos=('ANO','count'),
                Delta_médio_dias=('DELTA_DIAS','mean'))
           .round({'Delta_médio_dias': 0})
           .sort_values('Períodos', ascending=False))
    print(cob.to_string())


In [ ]:

if not dataset.empty:
    print("BLOCO 3 — Inventário de Contas por Demonstrativo")
    for pref in ['DRE','BPA','BPP','DFC']:
        cols = [c for c in dataset.columns if c.startswith(f'{pref}_')]
        if not cols: continue
        cob  = dataset[cols].notna().mean().sort_values(ascending=False)
        print(f"  {pref}: {len(cols)} contas | Top-5: {list(cob.head(5).index)}")


In [ ]:

if not dataset.empty:
    print("BLOCO 4 — Qualidade dos Dados (KPIs)")
    nulos = dataset[kpis_presentes].isnull().mean().sort_values(ascending=False)
    for kpi, v in nulos.items():
        sinal = "❌" if v > 0.5 else ("⚠️ " if v > 0.2 else "✅")
        print(f"  {sinal} {kpi:<25} {v:.0%}")
    df_num = dataset[kpis_presentes].dropna()
    if not df_num.empty:
        z = np.abs(stats.zscore(df_num, axis=0, nan_policy='omit'))
        print(f"  Outliers extremos |z|>5: {(z > 5).sum().sum()}")


In [ ]:

if not dataset.empty:
    print("BLOCO 5 — Estatísticas Descritivas dos KPIs")
    desc = dataset[kpis_presentes].describe().T
    print(desc[['count','mean','std','min','50%','max']].round(3).to_string())


In [ ]:

if not dataset.empty:
    print("BLOCO 6 — Targets Principais (Mediana por Setor, R$ mil)")
    tg = [c for c in ['DRE_3.01','DRE_3.11','EBITDA'] if c in dataset.columns]
    if tg:
        print(dataset.groupby('SETOR')[tg].median().round(0).to_string())


In [ ]:

if not dataset.empty:
    print("BLOCO 7 — Top-10 Correlações de Pearson com Receita Líquida")
    if 'DRE_3.01' in dataset.columns:
        corr = dataset[kpis_presentes + ['DRE_3.01']].corr()['DRE_3.01'].drop('DRE_3.01')
        print(corr.sort_values(key=abs, ascending=False).head(10).to_string())


In [ ]:

if not dataset.empty:
    print("BLOCO 8 — Evolução Temporal de Receita por Setor")
    if 'DRE_3.01' in dataset.columns and 'ANO' in dataset.columns:
        evol = (dataset.groupby(['SETOR','ANO'])['DRE_3.01']
                .median().unstack('SETOR').dropna(how='all'))
        print(evol.round(0).to_string())
        fig, ax = plt.subplots(figsize=(12, 5))
        evol.plot(ax=ax, marker='o')
        ax.set_title('Receita Líquida Mediana por Setor (R$ mil)')
        ax.set_xlabel('Ano'); ax.set_ylabel('R$ mil')
        ax.legend(loc='upper left'); plt.tight_layout()
        plt.savefig(PASTA_SAIDA / 'evol_receita_setor.png', dpi=150)
        plt.show()


In [ ]:

if not dataset.empty:
    print("BLOCO 9 — Inventário Final de KPIs para Modelagem")
    kpis_ok  = [k for k in kpis_presentes if dataset[k].notna().mean() > 0.5]
    kpis_exc = [k for k in kpis_presentes if k not in kpis_ok]
    print(f"  ✅ KPIs com >50% de cobertura ({len(kpis_ok)}):")
    for k in kpis_ok:
        print(f"      {k:<25} {dataset[k].notna().mean():.0%}")
    print(f"  ❌ KPIs excluídos da modelagem ({len(kpis_exc)}):")
    for k in kpis_exc:
        print(f"      {k:<25} {dataset[k].notna().mean():.0%}")


## Etapa 6 — Persistência: Parquet + CSV + Auditoria JSON

In [ ]:

if not dataset.empty:
    # ── Parquet (formato principal) ────────────────────────────────────────────
    # Vantagens sobre CSV: ~10× mais rápido para leitura, preserva dtypes
    # (datetime+tz, Int64, float64), compressão snappy eficiente.
    cam_pq = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
    dataset.to_parquet(cam_pq, index=False, compression='snappy', engine='pyarrow')
    logger.info("Parquet: %s | %d KB", cam_pq, cam_pq.stat().st_size // 1024)

    # ── CSV (opcional, compatibilidade Excel/Windows) ──────────────────────────
    cam_csv = PASTA_SAIDA / 'dataset_cvm_consolidado.csv'
    dataset.to_csv(cam_csv, index=False, encoding='utf-8-sig')
    logger.info("CSV    : %s | %d KB", cam_csv, cam_csv.stat().st_size // 1024)

    # ── Relatório de auditoria ─────────────────────────────────────────────────
    relatorio = {
        'timestamp_execucao'         : AGORA_LOCAL.isoformat(),
        'modo_rigoroso'              : MODO_RIGOROSO,
        'zips_processados'           : AUDITORIA['zips_processados'],
        'linhas_lidas_total'         : int(AUDITORIA['linhas_lidas_total']),
        'linhas_filtradas_anchor'    : int(AUDITORIA['linhas_filtradas_anchor']),
        'registros_descartados'      : int(AUDITORIA['registros_descartados']),
        'duplicatas_removidas_total' : int(AUDITORIA['duplicatas_removidas']),
        'erros_data_total'           : len(AUDITORIA['erros_data']),
        'datas_futuras_removidas'    : len(AUDITORIA['datas_futuras']),
        'inconsistencias_ano'        : len(AUDITORIA['inconsistencias_ano']),
        'periodos_irregulares'       : len(AUDITORIA['periodos_irregulares']),
        'alertas_merge_outer'        : len(AUDITORIA['alertas_merge_outer']),
        'erros_data_amostra'         : AUDITORIA['erros_data'][:20],
        'periodos_irregulares_amostra': AUDITORIA['periodos_irregulares'][:10],
        'dataset_shape'              : list(dataset.shape),
        'dataset_empresas'           : int(dataset['NOME_CIA'].nunique()),
        'dataset_anos'               : (sorted(dataset['ANO'].dropna()
                                              .astype(int).unique().tolist())
                                        if 'ANO' in dataset.columns else []),
        'kpis_cobertura'             : {
            k: round(float(dataset[k].notna().mean()), 4)
            for k in LISTA_KPIS if k in dataset.columns
        },
    }
    cam_audit = PASTA_SAIDA / 'auditoria_processamento.json'
    with open(cam_audit, 'w', encoding='utf-8') as f:
        json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)
    logger.info("Auditoria: %s", cam_audit)

    # ── Resumo terminal ────────────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("  RESUMO FINAL DO PIPELINE — Script 1")
    print("═" * 70)
    print(f"  Dataset          : {dataset.shape[0]:,} linhas × {dataset.shape[1]} colunas")
    print(f"  Empresas         : {dataset['NOME_CIA'].nunique()} / 25")
    print(f"  Anos cobertos    : {relatorio['dataset_anos']}")
    print(f"  Erros de data    : {len(AUDITORIA['erros_data'])}")
    print(f"  Datas futuras    : {len(AUDITORIA['datas_futuras'])}")
    print(f"  Períodos irreg.  : {len(AUDITORIA['periodos_irregulares'])}")
    print(f"  Duplicatas rem.  : {AUDITORIA['duplicatas_removidas']}")
    print(f"  Parquet          : {cam_pq}")
    print(f"  Auditoria        : {cam_audit}")
    print("═" * 70)
    print("  ✅ Pronto para Script 2 (cvm_preparacao)")
    print("═" * 70)
else:
    logger.error("Dataset vazio — nenhum arquivo salvo.")
